# 1.4 - Transformer Plumbing

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook builds the four wiring components that hold a transformer block together: normalization (LayerNorm and RMSNorm), positional encoding, residual connections, and finally the full Pre-LN block that combines all of them with attention and a feed-forward network. Each skill is a tiny, runnable experiment so you can see the plumbing behave, not just read about it.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install the numerical stack

Everything in this notebook runs on PyTorch tensors, so the first step is making sure numpy and torch are present. Nothing here is transformer-specific yet - it's just environment prep.

In [ ]:
!pip install -q numpy torch

**Explanation**

A single install command that pulls in numpy and torch quietly. Run once at the top; the rest of the notebook assumes both are available.

**How the code works - step by step**
- **`!pip install -q numpy torch`** - installs numpy and torch, `-q` keeps the log quiet.

**In one line:** get the numerical stack in place before any tensors are created.

**What you'll see:** (no output - environment prep)

## Imports - PyTorch layers and math

This cell pulls in the exact handful of tools every later cell reuses: the tensor library, the neural-network layer module, the functional API for attention, and Python's `math` for the log-scaled frequencies in positional encoding.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

**Explanation**

Pure import cell - no computation. It binds the four names the whole notebook leans on so nothing downstream needs to re-import.

**How the code works - step by step**
- **`torch`** - the core tensor library.
- **`torch.nn as nn`** - layer building blocks (`LayerNorm`, `RMSNorm`, `Linear`, `Sequential`, `Module`).
- **`torch.nn.functional as F`** - stateless ops, here `scaled_dot_product_attention`.
- **`math`** - used for `math.log(10000.0)` in the positional-encoding frequencies.

**In one line:** load torch, its layers, its functional API, and math.

**What you'll see:** (no output - environment prep)

## 1 - LayerNorm vs RMSNorm

Normalization keeps activations from exploding or vanishing as they pass through a deep stack. LayerNorm centers and scales each vector; RMSNorm skips the centering and only rescales. Running both on the same input is the fastest way to see the one difference between them - and why modern models (Llama, etc.) prefer the cheaper RMSNorm.

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
ln = nn.LayerNorm(4)
rms = nn.RMSNorm(4)
print(f'LayerNorm: {ln(x).round(decimals=3)}')
print(f'  mean: {ln(x).mean().item():.3f}, var: {ln(x).var(unbiased=False).item():.3f}')
print(f'RMSNorm:   {rms(x).round(decimals=3)}')
print(f'  mean: {rms(x).mean().item():.3f} (not zero), RMS: 1.0')

**Explanation**

A side-by-side measurement, not a model call: feed one 4-element vector through both norms and print the resulting statistics. The point is what each guarantees - LayerNorm forces mean 0 and variance 1, while RMSNorm only forces unit RMS and leaves the mean alone.

**How the code works - step by step**
- **`x = torch.tensor([[1.,2.,3.,4.]])`** - a single 4-dim input to normalize.
- **`ln = nn.LayerNorm(4)` / `rms = nn.RMSNorm(4)`** - two norm layers over the last dimension.
- **LayerNorm block** - prints the normalized vector, its mean, and its (biased) variance.
- **RMSNorm block** - prints the normalized vector and its mean, noting the RMS is 1.

**In one line:** LayerNorm centers *and* scales (mean 0, var 1); RMSNorm only scales (RMS 1, mean free).

**What you'll see:** The LayerNorm output has mean 0.000 and variance 1.000; the RMSNorm output has a non-zero mean while its root-mean-square is held at 1.0.

## 2 - Sinusoidal positional encoding

Attention has no built-in sense of order, so we add a fixed pattern that tells the model *where* each token sits. The original transformer uses sines and cosines at geometrically spaced frequencies. Building the matrix by hand shows how position and dimension combine into a unique signature per slot.

In [ ]:
def sinusoidal_pe(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    pos = torch.arange(0, seq_len).unsqueeze(1).float()
    div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe

pe = sinusoidal_pe(seq_len=8, d_model=16)
print(f'PE shape: {pe.shape}')
print(f'PE[0]: {pe[0][:4].round(decimals=3)}')
print(f'PE[7]: {pe[7][:4].round(decimals=3)}')

**Explanation**

A small generator function that returns a `seq_len x d_model` table of fixed (non-learned) position signals. Even dimensions get sines, odd dimensions get cosines, and the frequency shrinks across dimensions via the `10000` scaling - so each position gets a distinct, smoothly varying fingerprint.

**How the code works - step by step**
- **`pe = torch.zeros(seq_len, d_model)`** - the empty encoding table to fill.
- **`pos = torch.arange(...).unsqueeze(1)`** - a column of position indices 0..seq_len-1.
- **`div = torch.exp(... -log(10000)/d_model)`** - the geometric frequency schedule shared by sin and cos.
- **`pe[:, 0::2] = sin(...)` / `pe[:, 1::2] = cos(...)`** - fill even columns with sines, odd with cosines.
- **call + prints** - build an 8x16 encoding and inspect the first 4 values of positions 0 and 7.

**In one line:** stamp each position with a sin/cos pattern at descending frequencies.

**What you'll see:** Prints `PE shape: torch.Size([8, 16])`, then position 0's first four values (sin 0, cos 0 pattern - so 0, 1, 0, 1) and position 7's first four values showing a clearly different signature.

## 3 - Residual connections and gradient flow

This is the experiment that justifies the whole architecture. Stack 20 saturating (sigmoid) layers and watch the gradient at the input either die or survive depending on one thing: whether each layer adds its input back via a residual shortcut. It's a direct demonstration of the vanishing-gradient problem.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d, use_residual=True):
        super().__init__()
        self.use_residual = use_residual
        self.layer = nn.Sequential(nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        f = self.layer(x)
        return x + f if self.use_residual else f

def grad_at_input(use_residual, depth=20):
    torch.manual_seed(0)
    blocks = nn.Sequential(*[ResidualBlock(8, use_residual) for _ in range(depth)])
    x = torch.randn(1, 8, requires_grad=True)
    y = blocks(x).sum(); y.backward()
    return x.grad.norm().item()

print(f'20 layers, sigmoid, NO residual: grad norm = {grad_at_input(False):.4e}')
print(f'20 layers, sigmoid, WITH residual: grad norm = {grad_at_input(True):.4e}')

**Explanation**

A controlled A/B test, not a real training run. The same 20-layer stack is built twice - once with the skip connection, once without - and each time the gradient norm at the input is measured after a single backward pass. The residual path gives gradients a straight route back, so the input gradient stays healthy instead of shrinking toward zero.

**How the code works - step by step**
- **`ResidualBlock`** - a `Linear`+`Sigmoid` layer whose `forward` returns `x + f` when `use_residual`, else just `f`.
- **`grad_at_input`** - seeds RNG, stacks `depth` blocks, runs one forward+backward on a random input, and returns the input's gradient norm.
- **two prints** - call the harness with `use_residual=False` then `True` and report both gradient norms.

**In one line:** same 20 layers, same seed - the skip connection is the only difference in whether the gradient survives.

**What you'll see:** The no-residual stack prints a tiny gradient norm (vanished, e.g. on the order of 1e-6 or smaller), while the with-residual stack prints a much larger, healthy gradient norm - the shortcut rescues the signal.

## 4 - The full Pre-LN transformer block

Now assemble everything into the block that production models actually stack: normalize first (Pre-LN), run multi-head causal self-attention, add the residual, then normalize again and run a feed-forward network with its own residual. This is the exact plumbing pattern used by modern decoder LLMs.

In [ ]:
class PreLNBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=8):
        super().__init__()
        self.norm1 = nn.RMSNorm(d_model)
        self.norm2 = nn.RMSNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
    def forward(self, x):
        B, T, D = x.shape
        h = self.norm1(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        x = x + self.out(a)
        x = x + self.ffn(self.norm2(x))
        return x

block = PreLNBlock(d_model=128, n_heads=8)
x = torch.randn(2, 16, 128)
out = block(x)
print(f'output shape: {out.shape}')

**Explanation**

A complete, runnable transformer block wired from the earlier pieces. 'Pre-LN' means each sub-layer normalizes its input *before* the operation and adds the result back to the un-normalized stream - which is what keeps deep stacks stable. Attention uses PyTorch's fused `scaled_dot_product_attention` with a causal mask so each token only sees the past.

**How the code works - step by step**
- **`__init__`** - two `RMSNorm`s, a fused `qkv` projection, an output projection, and a 4x-wide GELU FFN; stores head count and per-head dim.
- **`norm1` + `qkv.chunk(3)`** - normalize, project to query/key/value, split into three.
- **`view(...).transpose(1,2)`** - reshape q/k/v into `(B, heads, T, d_head)` for multi-head attention.
- **`F.scaled_dot_product_attention(..., is_causal=True)`** - masked attention across positions.
- **`x = x + self.out(a)`** - merge heads, project, add the attention residual.
- **`x = x + self.ffn(self.norm2(x))`** - Pre-LN + FFN + second residual.
- **instantiate + run** - build a 128-dim, 8-head block and push a `(2, 16, 128)` batch through it.

**In one line:** norm -> attention -> add, then norm -> FFN -> add, shape preserved end to end.

**What you'll see:** Prints `output shape: torch.Size([2, 16, 128])` - the block returns the same shape it received, confirming the residual stream stays consistent through attention and the FFN.

You just wired a transformer block by hand: normalization pins activation scale, positional encoding injects order, residual connections keep gradients alive through depth, and the Pre-LN block stacks all three around attention and an FFN. The residual experiment (near-zero gradient without the skip, healthy gradient with it) is the single clearest reason deep transformers are trainable at all. Next module you'll assemble these blocks into full encoder and decoder models.